In [1]:
import psycopg
import pandas as pd

/Users/yanghede/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/yanghede/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.4' currently installed).
  from pandas.core import (


In [ ]:
# change to urs 
conninfo = "host=localhost port=5432 dbname=project user= password= "

In [3]:
def execute_sql(sql, params=None, many=False):
    with psycopg.connect(conninfo) as conn:
        with conn.cursor() as cur:
            if many:
                cur.executemany(sql, params)
            else:
                cur.execute(sql, params)
        conn.commit()

In [4]:
def query_df(sql, params=None):
    """
    For running queries. Returns pandas dataframe of query ouput.
    """
    with psycopg.connect(conninfo) as conn:
        with conn.cursor() as cur:
            cur.execute(sql, params)
            cols = [d.name for d in cur.description]
            return pd.DataFrame(cur.fetchall(), columns=cols)

### Create table

In [5]:
sql="""
CREATE TABLE IF NOT EXISTS works (
    work_id TEXT PRIMARY KEY,
    title TEXT,
    publication_year INT,
    cited_by_count INT,
    venue TEXT,
    abstract TEXT,
    embedding DOUBLE PRECISION[]
);

CREATE TABLE IF NOT EXISTS work_references (
    work_id TEXT NOT NULL,
    referenced_work_id TEXT NOT NULL,
    PRIMARY KEY (work_id, referenced_work_id)
);

CREATE TABLE IF NOT EXISTS concepts (
    concept_id TEXT PRIMARY KEY,
    concept_name TEXT
);

CREATE TABLE IF NOT EXISTS work_concepts (
    work_id TEXT NOT NULL,
    concept_id TEXT NOT NULL,
    score DOUBLE PRECISION,
    PRIMARY KEY (work_id, concept_id)
);

CREATE INDEX IF NOT EXISTS idx_work_references_work
ON work_references(work_id);

CREATE INDEX IF NOT EXISTS idx_work_references_ref
ON work_references(referenced_work_id);

CREATE INDEX IF NOT EXISTS idx_works_year
ON works(publication_year);
"""

In [29]:
execute_sql(sql)

In [30]:
execute_sql("""
ALTER TABLE works ADD COLUMN IF NOT EXISTS abstract TEXT;
ALTER TABLE works ADD COLUMN IF NOT EXISTS embedding DOUBLE PRECISION[];
""")

### OpenAlex 

In [6]:
import os
import re
import time
import requests

In [11]:
OPENALEX = "https://api.openalex.org"
PER_PAGE = 200  
SLEEP_SEC = 0.2 
API_KEY = os.getenv("OPENALEX_API_KEY")

In [ ]:

def oa_id(x: str) -> str:
    """Normalize OpenAlex IDs to short form like W..., T..., etc."""
    if not x:
        return None
    return x.replace("https://openalex.org/", "").strip()

In [ ]:

def clean_title(t: str) -> str:
    if t is None:
        return None
    t = re.sub(r"\s+", " ", t).strip()
    return t if t else None

In [ ]:
#Topic ID. e.g get_topic_id_by_search("natural language processing")-> T10123
def get_topic_id_by_search(query: str = "natural language processing") -> str:
    """
    Resolve a Topic ID using /topics?search=... (OpenAlex docs) :contentReference[oaicite:3]{index=3}
    """
    params = {"search": query, "per-page": 5}
    if API_KEY:
        params["api_key"] = API_KEY
    r = requests.get(f"{OPENALEX}/topics", params=params, timeout=30)
    r.raise_for_status()
    data = r.json()
    results = data.get("results", [])
    if not results:
        raise RuntimeError(f"No topic found for search='{query}'")
    return oa_id(results[0]["id"])
# 把 OpenAlex 提供的 abstract_inverted_index还原成正常的 abstract 文本。
# OpenAlex 的 abstract_inverted_index 长这样{"transformer": [0, 5],"model": [1],"for": [2],"nlp": [3]} 
# 真实 abstract 应该是transformer model for nlp transformer
def inverted_index_to_text(inv: dict) -> str:
    """
    OpenAlex gives abstract_inverted_index like:
      {"transformer":[0,5], "model":[1], ...}
    Convert it back to a normal abstract string.
    """
    if not inv:
        return None
    pairs = []
    for word, positions in inv.items():
        for p in positions:
            pairs.append((p, word))
    pairs.sort(key=lambda x: x[0])
    return " ".join(w for _, w in pairs)

In [ ]:
# 
def fetch_works_by_primary_topic(topic_id: str, n_target: int, year_min=2018, year_max=2024):
    """
    Fetch works filtered by primary_topic.id and publication_year.
    OpenAlex supports filtering works by topic fields like primary_topic.id :contentReference[oaicite:4]{index=4}
    """
    cursor = "*"
    out = []
    seen = set()

    while len(out) < n_target:
        params = {
            "filter": f"primary_topic.id:{topic_id},publication_year:>{year_min-1},publication_year:<{year_max+1}",
            "per-page": PER_PAGE,
            "cursor": cursor,
            "sort": "cited_by_count:desc",
        }
        if API_KEY:
            params["api_key"] = API_KEY

        r = requests.get(f"{OPENALEX}/works", params=params, timeout=60)
        r.raise_for_status()
        data = r.json()

        results = data.get("results", [])
        if not results:
            break

        for w in results:
            wid = oa_id(w.get("id"))
            if not wid or wid in seen:
                continue
            seen.add(wid)
            out.append(w)
            if len(out) >= n_target:
                break

        cursor = data.get("meta", {}).get("next_cursor")
        if not cursor:
            break

        time.sleep(SLEEP_SEC)

    return out

In [ ]:

# works_row = (wid, title, year, cited_by, venue, abstract)
# references_edges (list of (work_id, referenced_work_id))
# topic_rows (list of (concept_id, concept_name))
# work_topic_rows (list of (work_id, concept_id, score))
def extract_rows(work_obj):
    """
    Return:
      - works_row
      - references_edges (list of (work_id, referenced_work_id))
      - topic_rows (list of (concept_id, concept_name))
      - work_topic_rows (list of (work_id, concept_id, score))
    """
    wid = oa_id(work_obj.get("id"))

    title = clean_title(work_obj.get("title") or work_obj.get("display_name"))
    year = work_obj.get("publication_year")
    cited_by = work_obj.get("cited_by_count")
    abstract = inverted_index_to_text(work_obj.get("abstract_inverted_index"))


    venue = None
    try:
        venue = work_obj.get("primary_location", {}).get("source", {}).get("display_name")
    except Exception:
        venue = None
    venue = clean_title(venue)

    works_row = (wid, title, year, cited_by, venue, abstract)

    ref_ids = [oa_id(x) for x in (work_obj.get("referenced_works") or [])]
    ref_ids = [x for x in ref_ids if x]   
    references_edges = [(wid, rid) for rid in ref_ids]

    topic_rows = []
    work_topic_rows = []

    topics = work_obj.get("topics") or []
    for t in topics:
        tid = oa_id(t.get("id"))
        tname = clean_title(t.get("display_name"))
        score = t.get("score")
        if tid and tname:
            topic_rows.append((tid, tname))
        if tid and score is not None:
            work_topic_rows.append((wid, tid, float(score)))

    pt = work_obj.get("primary_topic") or {}
    ptid = oa_id(pt.get("id"))
    ptname = clean_title(pt.get("display_name"))
    if ptid and ptname:
        topic_rows.append((ptid, ptname))
        if not any(tid == ptid for (_, tid, _) in work_topic_rows):
            work_topic_rows.append((wid, ptid, 1.0))

    topic_rows = list({(a, b) for (a, b) in topic_rows})

    return works_row, references_edges, topic_rows, work_topic_rows

In [31]:
INSERT_WORKS = """
INSERT INTO works (work_id, title, publication_year, cited_by_count, venue, abstract)
VALUES (%s, %s, %s, %s, %s, %s)
ON CONFLICT (work_id) DO UPDATE SET
  title = EXCLUDED.title,
  publication_year = EXCLUDED.publication_year,
  cited_by_count = EXCLUDED.cited_by_count,
  venue = EXCLUDED.venue,
  abstract = COALESCE(EXCLUDED.abstract, works.abstract);
"""

INSERT_REFS = """
INSERT INTO work_references (work_id, referenced_work_id)
VALUES (%s, %s)
ON CONFLICT DO NOTHING;
"""

INSERT_CONCEPTS = """
INSERT INTO concepts (concept_id, concept_name)
VALUES (%s, %s)
ON CONFLICT (concept_id) DO UPDATE SET
  concept_name = EXCLUDED.concept_name;
"""

INSERT_WORK_CONCEPTS = """
INSERT INTO work_concepts (work_id, concept_id, score)
VALUES (%s, %s, %s)
ON CONFLICT (work_id, concept_id) DO UPDATE SET
  score = EXCLUDED.score;
"""

def batch_insert(rows, sql, batch_size=5000):
    for i in range(0, len(rows), batch_size):
        chunk = rows[i:i+batch_size]
        execute_sql(sql, chunk, many=True)

In [ ]:
#insert into psql 
def insert_paper(n_papers=5000, topic_search="natural language processing", year_min=2018, year_max=2024):
    topic_id = get_topic_id_by_search(topic_search)
    print(f"Using topic '{topic_search}' -> topic_id={topic_id}")

    works_raw = fetch_works_by_primary_topic(topic_id, n_target=n_papers, year_min=year_min, year_max=year_max)
    print(f"Fetched {len(works_raw)} works")

    works_rows = []
    refs_rows = []
    concept_rows = []
    work_concept_rows = []

    for w in works_raw:
        wr, rr, cr, wcr = extract_rows(w)
        if not wr[0] or not wr[1] or wr[2] is None:
            continue
        works_rows.append(wr)
        refs_rows.extend(rr)
        concept_rows.extend(cr)
        work_concept_rows.extend(wcr)

    refs_rows = list({(a, b) for (a, b) in refs_rows if a and b and a != b})
    concept_rows = list({(a, b) for (a, b) in concept_rows if a and b})
    work_concept_rows = list({(a, b, c) for (a, b, c) in work_concept_rows if a and b})

    print(f"Cleaned rows: works={len(works_rows)}, refs={len(refs_rows)}, concepts={len(concept_rows)}, work_concepts={len(work_concept_rows)}")

    batch_insert(works_rows, INSERT_WORKS, batch_size=2000)
    batch_insert(concept_rows, INSERT_CONCEPTS, batch_size=5000)
    batch_insert(refs_rows, INSERT_REFS, batch_size=10000)
    batch_insert(work_concept_rows, INSERT_WORK_CONCEPTS, batch_size=10000)

    print("✅ Done inserting into Postgres.")

    print(query_df("SELECT COUNT(*) AS n FROM works;"))
    print(query_df("SELECT COUNT(*) AS n FROM work_references;"))
    print(query_df("SELECT COUNT(*) AS n FROM concepts;"))
    print(query_df("SELECT COUNT(*) AS n FROM work_concepts;"))

In [ ]:
## insert 5000 papers 
insert_paper(n_papers=5000, topic_search="natural language processing", year_min=2018, year_max=2024)

Using topic 'natural language processing' -> topic_id=T10181
Fetched 5000 works
Cleaned rows: works=4971, refs=186232, concepts=173, work_concepts=13857
✅ Done inserting into Postgres.
      n
0  4972
        n
0  186256
     n
0  173
       n
0  13860


In [ ]:
# find the top match paper from psql 
def find_papers_by_title(title_query: str, limit: int = 10) -> pd.DataFrame:
    """
    Returns possible matches for a paper title.
    """
    return query_df(
        """
        SELECT work_id, title, publication_year, venue, cited_by_count
        FROM works
        WHERE title ILIKE %(q)s
        ORDER BY cited_by_count DESC NULLS LAST
        LIMIT %(limit)s;
        """,
        {"q": f"%{title_query}%", "limit": limit},
    )

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

embed_model = SentenceTransformer("all-MiniLM-L6-v2")  
def backfill_embeddings(batch_size: int = 512, where_extra_sql: str = ""):
    """
    Embed papers that have an abstract but no embedding yet.
    You can optionally pass extra filter, e.g. "AND publication_year >= 2018".
    """
    while True:
        df = query_df(f"""
        SELECT work_id, abstract
        FROM works
        WHERE abstract IS NOT NULL
          AND embedding IS NULL
          {where_extra_sql}
        LIMIT {batch_size}
        """)
        if df.empty:
            print("✅ No more missing embeddings.")
            return

        work_ids = df["work_id"].tolist()
        texts = df["abstract"].tolist()

        # normalized => dot product = cosine similarity
        embs = embed_model.encode(texts, normalize_embeddings=True, show_progress_bar=False)

        rows = [(emb.tolist(), wid) for emb, wid in zip(embs, work_ids)]
        execute_sql(
            "UPDATE works SET embedding = %s WHERE work_id = %s",
            rows,
            many=True
        )
        print(f"✅ Embedded {len(rows)} papers...")
        
backfill_embeddings(batch_size=512)

In [ ]:
import numpy as np
import pandas as pd

def recommend_papers_hybrid(
    paper_title: str,
    year_min: int = None,
    year_max: int = None,
    venue: str = None,
    topic: str = None,
    top_k: int = 10,
    candidate_pool: int = 800,
    min_shared_refs: int = 2,        
    w_jaccard: float = 4.0,          
    w_abs: float = 2.0,              
    w_cite: float = 0.2,             
    w_cocite: float = 1.0,           # 1. NEW: Weight for co-citations
) -> pd.DataFrame:
    
    matches = find_papers_by_title(paper_title, limit=20)
    if matches.empty:
        raise ValueError(f"No paper found with title containing: {paper_title}")
    input_work_id = matches.iloc[0]["work_id"]

    inp = query_df(
        """
        SELECT work_id, title, publication_year, venue, cited_by_count, embedding
        FROM works
        WHERE work_id = %(wid)s
        """,
        {"wid": input_work_id},
    )
    if inp.empty:
        raise ValueError("Input paper not found in works table.")
    if inp.iloc[0]["embedding"] is None:
        raise ValueError("Input paper has no embedding.")
    v = np.array(inp.iloc[0]["embedding"], dtype=np.float32)

    where_filters = []
    params = {"wid": input_work_id, "pool": candidate_pool, "min_shared": min_shared_refs}

    if year_min is not None:
        where_filters.append("w.publication_year >= %(year_min)s")
        params["year_min"] = year_min
    if year_max is not None:
        where_filters.append("w.publication_year <= %(year_max)s")
        params["year_max"] = year_max

    extra_where = ""
    if where_filters:
        extra_where = "AND " + " AND ".join(where_filters)

    # 2. NEW: The SQL query now includes the co_citations logic
    cand = query_df(f"""
    WITH input_refs AS (
      SELECT referenced_work_id
      FROM work_references
      WHERE work_id = %(wid)s
    ),
    input_ref_count AS (
      SELECT COUNT(*)::INT AS input_total_refs
      FROM input_refs
    ),
    cand_ref_count AS (
      SELECT work_id, COUNT(*)::INT AS cand_total_refs
      FROM work_references
      GROUP BY work_id
    ),
    coupling AS (
      SELECT r2.work_id AS candidate_id,
             COUNT(*)::INT AS shared_refs
      FROM input_refs ir
      JOIN work_references r2
        ON ir.referenced_work_id = r2.referenced_work_id
      WHERE r2.work_id <> %(wid)s
      GROUP BY r2.work_id
    ),
    co_citations AS (
      SELECT r2.referenced_work_id AS candidate_id,
             COUNT(*)::INT AS shared_citing_papers
      FROM work_references r1
      JOIN work_references r2 ON r1.work_id = r2.work_id
      WHERE r1.referenced_work_id = %(wid)s
        AND r2.referenced_work_id <> %(wid)s
      GROUP BY r2.referenced_work_id
    )
    SELECT
      w.work_id,
      w.title,
      w.publication_year,
      w.venue,
      w.cited_by_count,
      c.shared_refs,
      COALESCE(cc.shared_citing_papers, 0) AS shared_citing_papers,
      irc.input_total_refs,
      crc.cand_total_refs,
      (c.shared_refs::DOUBLE PRECISION /
         NULLIF(irc.input_total_refs + crc.cand_total_refs - c.shared_refs, 0)
      ) AS jaccard_refs,
      w.embedding
    FROM coupling c
    LEFT JOIN co_citations cc ON c.candidate_id = cc.candidate_id
    JOIN works w ON w.work_id = c.candidate_id
    CROSS JOIN input_ref_count irc
    JOIN cand_ref_count crc ON crc.work_id = c.candidate_id
    WHERE c.shared_refs >= %(min_shared)s
      AND w.embedding IS NOT NULL
      AND w.abstract IS NOT NULL
      AND TRIM(w.abstract) <> ''


      {extra_where} 
    ORDER BY
      jaccard_refs DESC NULLS LAST,
      c.shared_refs DESC,
      w.cited_by_count DESC NULLS LAST
    LIMIT %(pool)s;
    """, params)

    if cand.empty:
        raise ValueError("No candidates found.")

    E = np.stack(cand["embedding"].apply(lambda x: np.array(x, dtype=np.float32)).to_list(), axis=0)
    abs_sim = E @ v

    out = cand.drop(columns=["embedding"]).copy()
    out["abs_sim"] = abs_sim

    cites = out["cited_by_count"].fillna(0).astype(float).to_numpy()
    out["cite_term"] = np.log1p(cites)
    out["jaccard_refs"] = out["jaccard_refs"].fillna(0.0)

    # 3. NEW: Add the co-citation count into the final score
    out["final_score"] = (
        w_jaccard * out["jaccard_refs"].astype(float)
        + w_abs * out["abs_sim"].astype(float)
        + w_cite * out["cite_term"].astype(float)
        + w_cocite * out["shared_citing_papers"].astype(float)
    )

    out = out.sort_values(
        ["final_score", "jaccard_refs", "shared_refs", "cited_by_count"],
        ascending=[False, False, False, False],
        kind="mergesort"
    ).head(top_k)

    out.attrs["input_work_id"] = input_work_id
    return out

In [10]:
matches = find_papers_by_title("Attention Is All You Need")
print(matches)

recs = recommend_papers_hybrid(
    paper_title="Attention Is All You Need",
    year_min=2018,
    year_max=2024,
    venue=None,      # keep None until venue is well-populated
    topic=None,      # keep None until you confirm concept names
    top_k=10,
    candidate_pool=800,
    min_shared_refs=2,
)
print("input_work_id:", recs.attrs["input_work_id"])
display(recs)

       work_id                                              title  \
0  W3099527960  Attention Is All You Need for Chinese Word Seg...   

   publication_year venue  cited_by_count  
0              2020  None              30  
input_work_id: W3099527960


,work_id,title,publication_year,venue,cited_by_count,shared_refs,shared_citing_papers,input_total_refs,cand_total_refs,jaccard_refs,abs_sim,cite_term,final_score
0,W2905474383,Unsupervised Learning Helps Supervised Neural ...,2019,Proceedings of the AAAI Conference on Artifici...,18,17,0,44,45,0.236111,0.675441,2.944439,2.884214
4,W2788824299,Neural Networks Incorporating Dictionaries for...,2018,Proceedings of the AAAI Conference on Artifici...,53,10,0,44,25,0.169492,0.653071,3.988984,2.781904
8,W3035193825,Improving Chinese Word Segmentation with Wordh...,2020,None,94,9,0,44,53,0.102273,0.721655,4.553877,2.763175
137,W2946794439,Analyzing Multi-Head Self-Attention: Specializ...,2019,None,1009,4,0,44,42,0.048780,0.591932,6.917706,2.762528
42,W2963542740,Learning Deep Transformer Models for Machine T...,2019,None,614,5,0,44,44,0.060241,0.615629,6.421622,2.756547
1,W3103968805,A Joint Multiple Criteria Model in Transfer Le...,2020,None,22,13,0,44,36,0.194030,0.657732,3.135494,2.718683
3,W2964173597,Multiple Character Embeddings for Chinese Word...,2019,None,18,11,0,44,26,0.186441,0.665342,2.944439,2.665334
6,W2801805865,A Simple and Effective Neural Model for Joint ...,2018,IEEE/ACM Transactions on Audio Speech and Lang...,54,15,0,44,90,0.126050,0.625998,4.007333,2.557665
5,W2952662724,Is Word Segmentation Necessary for Deep Learni...,2019,arXiv (Cornell University),20,12,0,44,51,0.144578,0.679213,3.044522,2.545643
194,W2889326796,Understanding Back-Translation at Scale,2018,None,1007,5,0,44,75,0.043860,0.485364,6.915723,2.529312


In [11]:
def generate_explanation(row):
    """Translates the mathematical metrics into human-readable explanations."""
    reasons = []
    
    # Check the columns generated by your hybrid function
    if row.get('abs_sim', 0) > 0.65:
        reasons.append("Highly similar conceptual abstract")
    if row.get('shared_citing_papers', 0) > 0:
        
        reasons.append(f"Frequently co-cited ({int(row['shared_citing_papers'])} times)")
    if row.get('shared_refs', 0) > 5:
        reasons.append(f"Shares strong background ({int(row['shared_refs'])} shared references)")
    if row.get('cited_by_count', 0) > 500:
        reasons.append("Highly influential paper in the network")
        
    if not reasons:
        reasons.append("Solid overall similarity to your input paper")
        
    return " | ".join(reasons)
def recommend_ui(paper_title, year_min, year_max):
    """The wrapper function that connects the UI inputs to your backend code."""
    try:
        # 1. Fetch recommendations using your updated backend function
        df = recommend_papers_hybrid(
            paper_title=paper_title,
            year_min=int(year_min),
            year_max=int(year_max),
            top_k=10
        )
        
        # 2. Add the Explanation column
        df['Explanation'] = df.apply(generate_explanation, axis=1)
        
        # 3. Clean up the dataframe to make it look nice for the user
        display_df = df[['title', 'publication_year', 'cited_by_count', 'Explanation', 'final_score']]
        display_df.columns = ['Paper Title', 'Year', 'Citations', 'Why we recommend it', 'Score']
        
        # Round the score for cleaner display
        display_df['Score'] = display_df['Score'].round(3)
        
        return display_df
        
    except Exception as e:
        # If they type a paper that doesn't exist, this catches the error nicely
        return pd.DataFrame({"Error": [str(e)]})

In [12]:
recommend_ui("Attention Is All You Need", 2018, 2024)

/var/folders/lr/40t6g1_x16s8gwt15h_0f34m0000gp/T/ipykernel_50247/2779577288.py:39: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  display_df['Score'] = display_df['Score'].round(3)


,Paper Title,Year,Citations,Why we recommend it,Score
0,Unsupervised Learning Helps Supervised Neural ...,2019,18,Highly similar conceptual abstract | Shares st...,2.884
4,Neural Networks Incorporating Dictionaries for...,2018,53,Highly similar conceptual abstract | Shares st...,2.782
8,Improving Chinese Word Segmentation with Wordh...,2020,94,Highly similar conceptual abstract | Shares st...,2.763
137,Analyzing Multi-Head Self-Attention: Specializ...,2019,1009,Highly influential paper in the network,2.763
42,Learning Deep Transformer Models for Machine T...,2019,614,Highly influential paper in the network,2.757
1,A Joint Multiple Criteria Model in Transfer Le...,2020,22,Highly similar conceptual abstract | Shares st...,2.719
3,Multiple Character Embeddings for Chinese Word...,2019,18,Highly similar conceptual abstract | Shares st...,2.665
6,A Simple and Effective Neural Model for Joint ...,2018,54,Shares strong background (15 shared references),2.558
5,Is Word Segmentation Necessary for Deep Learni...,2019,20,Highly similar conceptual abstract | Shares st...,2.546
194,Understanding Back-Translation at Scale,2018,1007,Highly influential paper in the network,2.529


In [13]:
import gradio as gr
# --- Build the Gradio App Layout ---
with gr.Blocks(theme=gr.themes.Soft()) as app:
    gr.Markdown("# 📚 Academic Paper Recommender")
    gr.Markdown("Find what to read next based on semantic similarity, bibliographic coupling, and co-citations.")
    
    with gr.Row():
        with gr.Column(scale=1):
            title_input = gr.Textbox(
                label="Input Paper Title", 
                placeholder="e.g., Attention Is All You Need"
            )
            
            gr.Markdown("### Filter by Publication Year")
            # Here are the Slide Bars you requested! 
            year_min_slider = gr.Slider(minimum=2018, maximum=2024, value=2018, step=1, label="From Year")
            year_max_slider = gr.Slider(minimum=2018, maximum=2024, value=2024, step=1, label="To Year")
            
            submit_btn = gr.Button("Find Recommendations", variant="primary")
            
        with gr.Column(scale=2):
            output_table = gr.Dataframe(label="Recommended Papers", wrap=True)
            
    # Connect the button click to the wrapper function
    submit_btn.click(
        fn=recommend_ui,
        inputs=[title_input, year_min_slider, year_max_slider],
        outputs=output_table
    )

# Launch the app
app.launch(share=True)

ImportError: Pandas requires version '3.1.2' or newer of 'jinja2' (version '2.11.3' currently installed).

In [21]:
pip install --upgrade pandas

Note: you may need to restart the kernel to use updated packages.
